<a href="https://colab.research.google.com/github/antoniosolis4536-collab/MIAAD/blob/main/Sesion12_Evaluacion_Datos_Categoricos_274706.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Programación para Analítica Descriptiva y Predictiva**
**Maestría en Inteligencia Artificial y Analítica de Datos**

# Sesión 12: Evaluación — Limpieza y Transformación de Datos Categóricos

**Entrega individual**

- **Nombre**: ANTONIO ISMAEL SOLIS MARTINEZ
- **Matrícula** 274706

Esta evaluación aplica los tres temas de la Sesión 12 (errores tipográficos y valores inconsistentes, alta cardinalidad, tipos incorrectos) a un dataset que no se trabajó en clase: **Telco Customer Churn**.


No hay una única respuesta correcta en varias de las actividades — lo que se evalúa es que la conclusión esté respaldada por el código que la sustenta, no solo la conclusión en sí.

## Preparación

In [83]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


---
## Actividad 1 — Formato y valores inconsistentes (15 pts)

Revisa **todas** las columnas categóricas del dataset (no elijas solo una) en busca de variantes de formato (mayúsculas, espacios) que deberían normalizarse.

In [85]:
# 1.1 — Recorre todas las columnas categóricas con .value_counts() o .unique()
# para inspeccionar sus valores.

# Identificar columnas categoricas
columnas_categoricas = df.select_dtypes(include=['object']).columns.tolist()
columnas_categoricas.remove('customerID') #Identificador

for columna in columnas_categoricas:
    print(f"Valores únicos en la columna '{columna}':")
    print(df[columna].unique())
    print()


Valores únicos en la columna 'gender':
['Female' 'Male']

Valores únicos en la columna 'Partner':
['Yes' 'No']

Valores únicos en la columna 'Dependents':
['No' 'Yes']

Valores únicos en la columna 'PhoneService':
['No' 'Yes']

Valores únicos en la columna 'MultipleLines':
['No phone service' 'No' 'Yes']

Valores únicos en la columna 'InternetService':
['DSL' 'Fiber optic' 'No']

Valores únicos en la columna 'OnlineSecurity':
['No' 'Yes' 'No internet service']

Valores únicos en la columna 'OnlineBackup':
['Yes' 'No' 'No internet service']

Valores únicos en la columna 'DeviceProtection':
['No' 'Yes' 'No internet service']

Valores únicos en la columna 'TechSupport':
['No' 'Yes' 'No internet service']

Valores únicos en la columna 'StreamingTV':
['No' 'Yes' 'No internet service']

Valores únicos en la columna 'StreamingMovies':
['No' 'Yes' 'No internet service']

Valores únicos en la columna 'Contract':
['Month-to-month' 'One year' 'Two year']

Valores únicos en la columna 'PaperlessBi

**1.2 — Conclusión (responde aquí en Markdown):**

¿Encontraste alguna columna con inconsistencias de formato? Si sí, ¿cuál y qué código usarías para corregirla? Si no encontraste ninguna, dilo explícitamente — es una conclusión válida siempre que esté respaldada por lo que revisaste en 1.1.

_Tu respuesta:_ Revisé todas las columnas y no encontré ninguna con errores de mayúsculas o espacios de más. Todos los valores están escritos igual siempre (por ejemplo, gender solo tiene Female y Male, sin variantes). Creo que esto es porque el dataset viene de un sistema con opciones fijas (como un menú desplegable), no de texto libre donde la gente escribe lo que quiera.

---
## Actividad 2 — Valores inválidos (20 pts)

Varias columnas de este dataset (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) tienen un tercer valor además de `'Yes'`/`'No'`: `'No internet service'`. De forma similar, `MultipleLines` tiene `'No phone service'`.

In [86]:
df['OnlineSecurity'].value_counts()

,count
OnlineSecurity,
No,3498
Yes,2019
No internet service,1526


In [87]:
# 2.1 — Verifica: ¿las filas con 'No internet service' en OnlineSecurity
# coinciden con las filas donde InternetService == 'No'?
# (pista: cruza ambas columnas con pd.crosstab o filtrando)

tabla_cruzada = pd.crosstab(df['OnlineSecurity'], df['InternetService'])
print(tabla_cruzada)


InternetService       DSL  Fiber optic    No
OnlineSecurity                              
No                   1241         2257     0
No internet service     0            0  1526
Yes                  1180          839     0


**2.2 — Conclusión (responde aquí en Markdown):**

¿`'No internet service'` es un valor inválido (como `Absurd`/`YOLO` en la sesión de clase) o es una categoría legítima? Justifica tu respuesta con lo que verificaste en 2.1. ¿Tomarías alguna acción sobre esta columna, o la dejarías tal cual?

_Tu respuesta:_ No internet service sí es un valor válido, no un error. Al cruzar la columna con InternetService, vi que todos los que tienen 'No internet service' en OnlineSecurity son exactamente los mismos que no tienen internet contratado (InternetService == 'No'). Tiene sentido: si no tienes internet, obviamente no puedes tener seguridad en línea. No le haría nada a esta columna, está bien como está.

---
## Actividad 3 — Alta cardinalidad (25 pts)

In [88]:
# 3.1 — Calcula .nunique() para TODAS las columnas del dataset (no solo las categóricas)
# y la razón (valores únicos / total de filas) para cada una.

for columna in df.columns:
    valores_unicos = df[columna].nunique()
    total_filas = len(df)
    razon = valores_unicos / total_filas
    print(f"La columna '{columna}' tiene {valores_unicos} valores únicos y una razón de {razon:.2f} para cada valor único.")


La columna 'customerID' tiene 7043 valores únicos y una razón de 1.00 para cada valor único.
La columna 'gender' tiene 2 valores únicos y una razón de 0.00 para cada valor único.
La columna 'SeniorCitizen' tiene 2 valores únicos y una razón de 0.00 para cada valor único.
La columna 'Partner' tiene 2 valores únicos y una razón de 0.00 para cada valor único.
La columna 'Dependents' tiene 2 valores únicos y una razón de 0.00 para cada valor único.
La columna 'tenure' tiene 73 valores únicos y una razón de 0.01 para cada valor único.
La columna 'PhoneService' tiene 2 valores únicos y una razón de 0.00 para cada valor único.
La columna 'MultipleLines' tiene 3 valores únicos y una razón de 0.00 para cada valor único.
La columna 'InternetService' tiene 3 valores únicos y una razón de 0.00 para cada valor único.
La columna 'OnlineSecurity' tiene 3 valores únicos y una razón de 0.00 para cada valor único.
La columna 'OnlineBackup' tiene 3 valores únicos y una razón de 0.00 para cada valor único

**3.2 — Conclusión (responde aquí en Markdown):**

¿Qué columna(s) tienen alta cardinalidad? Para la columna con mayor cardinalidad: ¿por qué nunca deberías usarla como variable predictora en un modelo, incluso si la codificaras? (relaciona tu respuesta con lo discutido en clase sobre identificadores únicos)

_Tu respuesta:_ customerID es la que tiene más valores únicos, casi uno por cliente. No serviría como variable para un modelo porque cada ID solo aparece una vez — el modelo no puede aprender nada de un valor que nunca se repite, solo estaría memorizando quién es quién, no encontrando un patrón.

**3.3 — Agrupación "Top 10 + Otros"**

En clase agrupaste `country` de Netflix Titles en sus 10 categorías más frecuentes + `'Otros'`, reduciendo su cardinalidad. Aplica la misma técnica aquí sobre la columna de mayor cardinalidad que identificaste en 3.1 (pista: `.value_counts().head(10)`, luego `.where()` + `.isin()`, igual que en el notebook de clase).

In [89]:
# Aplica el agrupamiento Top 10 + Otros sobre la columna de mayor cardinalidad
top10_ids = df['customerID'].value_counts().head(10)
categorias_top10 = top10_ids.index.tolist()
df['customerID'] = df['customerID'].where(df['customerID'].isin(categorias_top10), 'Otros')

df['customerID'].value_counts()



,count
customerID,
Otros,7033
7590-VHVEG,1
5575-GNVDE,1
9837-FWLCH,1
1699-HPSBG,1
7203-OYKCT,1
1035-IPQPU,1
7398-LXGYX,1
2823-LKABH,1


**3.4 — Conclusión (responde aquí en Markdown):**

Después de agrupar, ¿la columna resultante te parece útil para un modelo? Compara este caso con el de `country` en el notebook de clase: ¿por qué agrupar en "Top 10 + Otros" funciona bien para una variable como `country`, pero no resuelve el problema real de la columna que agrupaste aquí?

_Tu respuesta:_ Aunque agrupé en Top 10 + Otros, la columna sigue sin servir. Con country en el ejemplo de clase, el Top 10 sí agarraba una buena parte de los datos reales (Estados Unidos solo ya era un chorro de filas). Aquí no — cada ID es único, entonces el "Top 10" que salió fue casi al azar y casi todo terminó en "Otros". El problema no es que falte agrupar bien, es que esta columna nunca debió tratarse como categoría desde el principio.

---
## Actividad 4 — Tipos de dato (30 pts)

In [90]:
df['TotalCharges'].dtype

dtype('O')

**4.1 — Investiga (responde en Markdown):**

`TotalCharges` contiene valores numéricos (montos en dólares), pero pandas la cargó como `object`, no como `float`. Investiga por qué — revisa si hay algún valor que no se vea como un número normal.

_Tu respuesta:_ Al buscar qué valores no se podían convertir a número, encontré que hay filas con solo un espacio en blanco en vez de un número. Todas esas filas son de clientes con tenure en 0, o sea, clientes que apenas entraron y todavía no les cobran nada — por eso el campo se quedó vacío en vez de poner 0. Como hay al menos un valor que no es número, pandas trata toda la columna como texto.

In [91]:
# 4.2 — Corrige el tipo de TotalCharges.
# Pista: pd.to_numeric() con el parámetro errors= te puede ayudar a identificar
# o manejar los valores problemáticos que encontraste en 4.1.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(df['TotalCharges'].dtype)
print('valores nulos despues de convertir:', df['TotalCharges'].isnull().sum())


float64
valores nulos despues de convertir: 11


In [93]:
# 4.3 — Convierte a category las columnas categóricas que, según lo que calculaste
# en la Actividad 3, tengan cardinalidad baja y valores fijos.
# Verifica con .dtypes que el cambio se aplicó correctamente.
columnas_a_categoria = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod', 'Churn'
]

for col in columnas_a_categoria:
  df[col] = df[col].astype('category')

df[columnas_a_categoria].dtypes

,0
gender,category
SeniorCitizen,category
Partner,category
Dependents,category
PhoneService,category
MultipleLines,category
InternetService,category
OnlineSecurity,category
OnlineBackup,category
DeviceProtection,category


---
## Reflexión final (10 pts)

Con base en las 4 actividades anteriores, responde:

1. De las alertas que detectaste (formato, valores inválidos, cardinalidad, tipos), ¿cuál te pareció más fácil de decidir y cuál más difícil? ¿Por qué?
2. Si tuvieras que entregar este dataset ya "perfilado" a un compañero para que construya un modelo predictivo, ¿qué le dirías sobre `customerID` y sobre `TotalCharges`?

_Tu respuesta:_
1) Lo más fácil fue la parte de formato, porque el dataset ya estaba limpio y no había mucho que pensar. Lo más difícil fue lo de customerID, porque ahí no hay un "arreglo" tienes que aceptar que esa columna simplemente no sirve para el modelo.

2) Diría que quite customerID desde el principio, y no deberiamos meterlo en el modelo. Y de TotalCharges, venía como texto por unos espacios en blanco de clientes nuevos, ya la corregí a número, pero eso dejó algunos vacíos (NaN) que hay que decidir qué hacer con ellos, lo más lógico sería ponerles 0.

---
## Rúbrica de evaluación

| Actividad | Puntos | Criterio |
|---|---|---|
| 1. Formato y valores inconsistentes | 15 | Revisó todas las columnas categóricas (no solo una); código comentado y conclusión (1.2) respaldada por lo que se observó, no solo afirmada |
| 2. Valores inválidos | 20 | Verificó la relación entre columnas antes de concluir; la conclusión (2.2) justifica con evidencia, no solo con intuición |
| 3. Alta cardinalidad | 25 | Calcula `.nunique()` para todas las columnas; aplica correctamente el agrupamiento Top 10 + Otros; la conclusión (3.4) explica por qué agrupar no resuelve el problema de un identificador único |
| 4. Tipos de dato | 30 | Identifica la causa raíz del `dtype` incorrecto (4.1); corrige `TotalCharges` sin perder información; conversión a `category` justificada por cardinalidad, no aplicada al azar |
| Reflexión final | 10 | Conecta las 4 actividades entre sí; no es una respuesta genérica o intercambiable con cualquier dataset |
| **Total** | **100** | |

**Nota sobre las conclusiones:** cada actividad tiene su propia pregunta de conclusión (1.2, 2.2, 3.4, 4.1) — esas respuestas se califican como parte de la actividad correspondiente, no solo la Reflexión final. Comenta tu código donde tomes una decisión (por ejemplo, por qué elegiste cierto umbral o cierta corrección) — el comentario también cuenta dentro del puntaje de cada actividad.